[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C42_Learning_Theory_Course/02_optimization_theory/02_讲解.ipynb)

# 02 · 优化理论（实测收敛率对拍理论速率）

目标：把 **GD 的 O(1/t) 与强凸线性率**、**SGD 的噪声与 O(1/t)**、**动量的 √κ 加速** 用纯 numpy 跑出实测曲线，并 `assert` 它逐点压在理论上界之下。

路线：构造可控二次目标 → GD 凸 O(1/t) → GD 强凸线性收敛 → 步长与条件数 → SGD 噪声球 + 1/t 步长(=在线均值) → 动量 √κ 加速 → logistic 回归上的 GD → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 本章最容易『看见理论』：实测损失曲线会严丝合缝压在理论速率之下。

In [ ]:
import numpy as np
import math
rng = np.random.default_rng(1)
# 可控二次目标 f(x)=0.5 x^T A x, A 正定, 最优在 0, f*=0。L=lambda_max, mu=lambda_min。
def make_quadratic(d=20, cond=None, seed=1):
    g = np.random.default_rng(seed)
    if cond is None:
        B = g.standard_normal((d, d)); A = B @ B.T / d + 0.05*np.eye(d)
    else:
        # 指定条件数：特征值在 [1, cond] 对数均布
        Q, _ = np.linalg.qr(g.standard_normal((d, d)))
        evals = np.logspace(0, np.log10(cond), d)
        A = (Q * evals) @ Q.T
    A = (A + A.T) / 2
    return A
A = make_quadratic(20, seed=1)
ev = np.linalg.eigvalsh(A); L, mu = ev.max(), ev.min()
print(f'd=20 二次目标: L={L:.3f}, mu={mu:.4f}, kappa=L/mu={L/mu:.1f}')
f    = lambda x: 0.5 * x @ A @ x
grad = lambda x: A @ x

## 1 · GD 凸 + 光滑：实测损失 $\le \dfrac{L\|x_0-x^\*\|^2}{2t}$

步长 $\eta=1/L$。理论 $f(x_t)-f^\*\le\frac{L\|x_0\|^2}{2t}$（这里 $x^\*=0,f^\*=0$）。验证实测逐点压在上界下。

In [ ]:
x0 = rng.standard_normal(20)
x = x0.copy(); T = 1000; fs = []
for t in range(T):
    x = x - (1/L) * grad(x); fs.append(f(x))
fs = np.array(fs)
ub = L * np.dot(x0, x0) / (2 * np.arange(1, T+1))
for t in [1, 10, 100, 1000]:
    print(f't={t:5d}  f={fs[t-1]:.3e}  O(1/t)上界={ub[t-1]:.3e}  <=? {fs[t-1] <= ub[t-1]}')
assert np.all(fs <= ub + 1e-12), 'GD 凸 O(1/t) 上界必须对所有 t 成立'
print('\n✅ 实测损失逐点压在 L||x0||^2/(2t) 之下 —— O(1/t) 被钉死。')

## 2 · GD 强凸：实测距离 $\le (1-\mu/L)^t\|x_0-x^\*\|^2$（线性收敛）

同样步长 $1/L$，强凸下距离平方几何衰减。验证实测距离逐点压在 $(1-\mu/L)^t\|x_0\|^2$ 之下，并核对收敛因子。

In [ ]:
x = x0.copy(); T = 300; dists = []
for t in range(T):
    x = x - (1/L) * grad(x); dists.append(np.dot(x, x))
dists = np.array(dists)
rate = 1 - mu/L
ub2 = (rate ** np.arange(1, T+1)) * np.dot(x0, x0)
for t in [1, 50, 150, 300]:
    print(f't={t:5d}  ||x_t||^2={dists[t-1]:.3e}  上界={ub2[t-1]:.3e}  <=? {dists[t-1] <= ub2[t-1]}')
print(f'收敛因子 (1-mu/L) = {rate:.4f}  (kappa={L/mu:.1f})')
assert np.all(dists <= ub2 + 1e-12), '强凸线性收敛上界必须成立'
print('✅ 距离平方几何衰减，逐点压在 (1-mu/L)^t 之下 —— 线性(指数)收敛被钉死。')

## 3 · 条件数决定速度 + 动量的 √κ 加速

在一个**病态**二次（$\kappa\approx400$）上，对比 GD 与 Polyak 重球法到达 $10^{-6}$ 所需迭代数。
理论：GD $\propto\kappa$，重球 $\propto\sqrt\kappa$，故加速比 $\approx\sqrt\kappa$。

In [ ]:
Ai = make_quadratic(20, cond=400, seed=3)
ev = np.linalg.eigvalsh(Ai); Li, mui = ev.max(), ev.min(); kappa = Li/mui
gi = lambda x: Ai @ x
z0 = rng.standard_normal(20); tol = 1e-6 * np.dot(z0, z0)
def iters_to_tol(step_fn, T=20000):
    seq = step_fn(T)
    idx = np.where(seq < tol)[0]
    return idx[0] + 1 if len(idx) else T
def gd_run(T):
    x = z0.copy(); out = []
    for _ in range(T): x = x - (1/Li)*gi(x); out.append(np.dot(x,x))
    return np.array(out)
def heavy_ball(T):
    beta = ((np.sqrt(Li)-np.sqrt(mui))/(np.sqrt(Li)+np.sqrt(mui)))**2
    alpha = (2/(np.sqrt(Li)+np.sqrt(mui)))**2
    xp = z0.copy(); x = z0.copy(); out = []
    for _ in range(T):
        xn = x - alpha*gi(x) + beta*(x - xp); xp = x; x = xn; out.append(np.dot(x,x))
    return np.array(out)
ig = iters_to_tol(gd_run); ih = iters_to_tol(heavy_ball)
print(f'kappa={kappa:.0f}, sqrt(kappa)={np.sqrt(kappa):.1f}')
print(f'GD 到 1e-6: {ig} 步;  重球: {ih} 步;  加速比 = {ig/ih:.1f}x')
assert ih < ig, '重球法应更快'
assert ig/ih > 3.0, '病态问题上加速比应是明显的多倍'
print('✅ 加速比与 sqrt(kappa) 同量级 —— 动量把 kappa 改善成 sqrt(kappa) 被钉死。')

## 4 · SGD 恒定步长：停在半径 $\sim\sqrt{\eta\sigma^2}$ 的噪声球

对二次目标加梯度噪声（每步 $\nabla f(x)+\xi$，$\xi\sim\mathcal N(0,\sigma^2 I)$），恒定步长 $\eta$。SGD 不收敛到 0，而是停在期望次优 $\sim\eta\sigma^2$ 的球内。验证：步长翻倍，稳态误差约翻倍。

In [ ]:
def sgd_const(eta, noise, T=8000, seed=0):
    g = np.random.default_rng(seed); x = z0.copy()
    tail = []
    for t in range(T):
        xi = g.standard_normal(20) * noise
        x = x - eta * (Ai @ x + xi)
        if t > T - 2000: tail.append(f_i(x))
    return np.mean(tail)
f_i = lambda x: 0.5 * x @ Ai @ x
noise = 1.0
e1 = sgd_const(0.5/Li, noise); e2 = sgd_const(1.0/Li, noise)
print(f'稳态次优 f-f*:  eta=0.5/L -> {e1:.4f},  eta=1.0/L -> {e2:.4f}')
print(f'步长翻倍 -> 稳态误差比 = {e2/e1:.2f} (理论 ~2, 噪声球 ∝ eta)')
assert e2 > e1, '步长越大噪声球越大'
assert 1.3 < e2/e1 < 3.0, '稳态误差应大致正比于步长'
print('✅ 恒定步长 SGD 停在 ∝ eta 的噪声球 —— 要继续下降须衰减步长。')

## 5 · SGD $1/(\mu t)$ 步长 = 在线均值：$\mathbb E[(x_T-x^\*)^2]=\sigma^2/T$

最小化 $\tfrac12\mathbb E[(x-z)^2]$，$z\sim\mathcal N(m,\sigma^2)$（$\mu=1$）。SGD 步长 $1/t$ 迭代 $x_{t+1}=x_t-\tfrac1t(x_t-z_t)$ **恰好**是在线样本均值，故误差精确 $\sigma^2/T$。多次重复实测其期望并对拍。

In [ ]:
m_true, sigma = 3.0, 2.0
def sgd_mean(T, seed):
    g = np.random.default_rng(seed); x = 0.0
    for t in range(1, T+1):
        z = g.normal(m_true, sigma)
        x = x - (1.0/t) * (x - z)        # = 在线均值
    return x
T = 2000
errs = [(sgd_mean(T, s) - m_true)**2 for s in range(400)]
print(f'E[(x_T-x*)^2] 实测 (400次) = {np.mean(errs):.5f}')
print(f'理论 sigma^2/T            = {sigma**2/T:.5f}')
assert abs(np.mean(errs)/(sigma**2/T) - 1) < 0.2, 'SGD 1/t 步长误差应 = sigma^2/T'
print('✅ 1/(mu t) 步长 SGD 等于在线均值，误差精确 sigma^2/T —— O(1/t) 强凸率被钉死。')

## 6 · logistic 回归上的 GD（非二次的凸光滑）

logistic 损失也是凸 + 光滑（光滑常数 $L=\tfrac1{4n}\lambda_{\max}(X^\top X)$）。验证步长 $1/L$ 的 GD 收敛（损失单调下降趋于最优）。

In [ ]:
ng = np.random.default_rng(4); n, d = 200, 10
Xd = ng.standard_normal((n, d)); w_t = ng.standard_normal(d)
yb = (ng.random(n) < 1/(1+np.exp(-(Xd@w_t)))).astype(float)   # 伯努利标签(有噪声 -> 不可分)
def logloss(w):
    z = Xd @ w
    return np.mean(np.logaddexp(0, z) - yb*z)
def loggrad(w):
    p = 1/(1+np.exp(-(Xd@w)))
    return Xd.T @ (p - yb) / n
L_log = np.linalg.eigvalsh(Xd.T @ Xd).max() / (4*n)
w = np.zeros(d); losses = []
for t in range(3000):
    w = w - (1/L_log)*loggrad(w); losses.append(logloss(w))
losses = np.array(losses)
print(f'logistic L={L_log:.4f}; 初始损失={np.logaddexp(0,0):.4f}, 末损失={losses[-1]:.4f}')
# 凸 + 光滑步长 1/L -> 损失单调不增
assert np.all(np.diff(losses) <= 1e-9), '步长 1/L 的 GD 在凸光滑上损失应单调不增'
assert losses[-1] < losses[0], '损失应下降'
print('✅ logistic(凸光滑) 上 GD 步长 1/L 损失单调下降收敛 —— descent lemma 生效。')

---
## ✏️ 练习 1：凸 GD 达到 ε 所需迭代数

凸光滑 GD：$f(x_t)-f^\*\le\frac{L D^2}{2t}$（$D=\|x_0-x^\*\|$）。实现 `iters_convex(L, D, eps)` 返回保证 $f(x_t)-f^\*\le\varepsilon$ 的最小整数 $t$。

In [ ]:
import math
def iters_convex(L, D, eps):
    # TODO: 解 L D^2/(2t) <= eps -> t >= L D^2/(2 eps); 返回 ceil
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
t1 = iters_convex(10.0, 2.0, 0.01)
assert t1 == math.ceil(10.0*4/(2*0.01)), '公式不符'
# eps 减半 -> 迭代数翻倍 (O(1/eps))
assert iters_convex(10,2,0.005)/iters_convex(10,2,0.01) == 2
print(f'L=10,D=2,eps=0.01 -> t>={t1};  eps 减半 -> {iters_convex(10,2,0.005)} (×2)')
print('✅ 练习 1 通过：凸 GD 是 O(1/eps)')

## ✏️ 练习 2：强凸线性收敛迭代数

强凸 GD（步长 $1/L$）：$\|x_t-x^\*\|^2\le(1-1/\kappa)^t\|x_0-x^\*\|^2$。实现 `iters_strongly_convex(kappa, D, eps)` 返回保证 $\|x_t-x^\*\|^2\le\varepsilon$ 的最小整数 $t$。

In [ ]:
def iters_strongly_convex(kappa, D, eps):
    # TODO: 解 (1-1/kappa)^t D^2 <= eps -> t >= ln(eps/D^2)/ln(1-1/kappa); 返回 ceil
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
t2 = iters_strongly_convex(100, 1.0, 1e-6)
import math
expect = math.ceil(math.log(1e-6) / math.log(1 - 1/100))
assert t2 == expect, '公式不符'
# kappa 增大 -> 迭代数约线性增大 (O(kappa log 1/eps))
r = iters_strongly_convex(400,1.0,1e-6) / iters_strongly_convex(100,1.0,1e-6)
assert 3.0 < r < 5.0, 'kappa 增 4 倍, 迭代数约增 4 倍'
print(f'kappa=100 -> t>={t2};  kappa 增 4 倍 -> 迭代数 ×{r:.1f} (O(kappa))')
print('✅ 练习 2 通过：强凸是 O(kappa·log 1/eps)')

## ✏️ 练习 3：SGD 方差–步长权衡

复用 worked 4 的恒定步长 SGD。实现 `noise_ball(eta, noise)` 返回稳态次优（末 2000 步的平均 $f-f^\*$）。验证：固定噪声，稳态误差随步长**单调增**。

In [ ]:
def noise_ball(eta, noise, T=8000, seed=0):
    # TODO: 复用 worked 4 的 sgd_const 逻辑（梯度 Ai@x + 高斯噪声*noise, 步长 eta）,
    #       返回末 2000 步 f_i(x) 的平均
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
vals = [noise_ball(s/Li, 1.0) for s in [0.25, 0.5, 1.0]]
print('eta=[0.25,0.5,1.0]/L -> 稳态次优 =', [f'{v:.4f}' for v in vals])
assert vals[0] < vals[1] < vals[2], '步长越大噪声球越大（稳态误差单调增）'
print('✅ 练习 3 通过：SGD 稳态误差随步长单调增（方差-步长权衡）')

## ✏️ 练习 4：动量加速比 ≈ √κ

实现 `accel_ratio(cond)`：在条件数 `cond` 的二次上返回 `GD 到 1e-6 的迭代数 / 重球到 1e-6 的迭代数`。验证该比值随 $\sqrt{\text{cond}}$ 增长（cond 翻 4 倍，比值约翻 2 倍）。

In [ ]:
def accel_ratio(cond, seed=3):
    # TODO: make_quadratic(20, cond) -> 算 L,mu -> GD(步长1/L) 与 重球(Polyak 最优参数)
    #       各自到 1e-6*||z0||^2 的迭代数, 返回 GD步数/重球步数。z0 用固定种子。
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
r100 = accel_ratio(100); r400 = accel_ratio(400)
print(f'cond=100 加速比={r100:.1f};  cond=400 加速比={r400:.1f}')
assert r100 > 2 and r400 > r100, '加速比随 sqrt(cond) 增长'
assert 1.3 < r400/r100 < 3.0, 'cond 翻 4 倍, 加速比约翻 2 倍 (sqrt)'
print('✅ 练习 4 通过：动量加速比 ∝ sqrt(kappa)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def iters_convex(L, D, eps):
    return math.ceil(L * D**2 / (2 * eps))

In [ ]:
# 练习 2
def iters_strongly_convex(kappa, D, eps):
    return math.ceil(math.log(eps / D**2) / math.log(1 - 1/kappa))

In [ ]:
# 练习 3
def noise_ball(eta, noise, T=8000, seed=0):
    g = np.random.default_rng(seed); x = z0.copy(); tail = []
    for t in range(T):
        xi = g.standard_normal(20) * noise
        x = x - eta * (Ai @ x + xi)
        if t > T - 2000: tail.append(0.5 * x @ Ai @ x)
    return np.mean(tail)

In [ ]:
# 练习 4
def accel_ratio(cond, seed=3):
    A2 = make_quadratic(20, cond=cond, seed=seed)
    ev = np.linalg.eigvalsh(A2); L2, mu2 = ev.max(), ev.min()
    g2 = lambda x: A2 @ x
    z = np.random.default_rng(99).standard_normal(20); tol = 1e-6*np.dot(z,z)
    def gd_it():
        x = z.copy()
        for t in range(50000):
            x = x - (1/L2)*g2(x)
            if np.dot(x,x) < tol: return t+1
        return 50000
    def hb_it():
        beta = ((np.sqrt(L2)-np.sqrt(mu2))/(np.sqrt(L2)+np.sqrt(mu2)))**2
        alpha = (2/(np.sqrt(L2)+np.sqrt(mu2)))**2
        xp = z.copy(); x = z.copy()
        for t in range(50000):
            xn = x - alpha*g2(x) + beta*(x-xp); xp = x; x = xn
            if np.dot(x,x) < tol: return t+1
        return 50000
    return gd_it() / hb_it()

---
## 🧪 真实数据胶囊：在真实数据(乳腺癌)的 logistic 回归上对拍条件数与收敛

用真实数据集（sklearn 乳腺癌；失败回退到真实形状的合成数据）构造 logistic 回归，算其 Hessian 在最优附近的条件数，并验证 GD 收敛、步长 $1/L$ 损失单调下降。

In [ ]:
try:
    from sklearn.datasets import load_breast_cancer
    dd = load_breast_cancer()
    Xc = dd.data.astype(float); yc = dd.target.astype(float)
    Xc = (Xc - Xc.mean(0)) / (Xc.std(0) + 1e-9)
    Xc = np.hstack([Xc, np.ones((len(Xc),1))])   # 截距
    src = 'sklearn breast_cancer (真实)'
except Exception as e:
    g = np.random.default_rng(0); Xc = g.standard_normal((569, 31)); Xc[:,-1]=1.0
    yc = (g.random(569) < 0.5).astype(float); src = f'回退合成: {type(e).__name__}'
nC, dC = Xc.shape
print(f'数据来源: {src}; 形状 {Xc.shape}')

In [ ]:
def ll_grad(w):
    p = 1/(1+np.exp(-(Xc@w))); return Xc.T@(p-yc)/nC
def ll_loss(w):
    z = Xc@w; return np.mean(np.logaddexp(0,z) - yc*z)
L_C = np.linalg.eigvalsh(Xc.T@Xc).max()/(4*nC)   # logistic 光滑常数上界
w = np.zeros(dC); hist = []
for t in range(4000):
    w = w - (1/L_C)*ll_grad(w); hist.append(ll_loss(w))
hist = np.array(hist)
print(f'logistic 光滑常数 L={L_C:.4f}; 损失 {hist[0]:.4f} -> {hist[-1]:.4f}')
assert np.all(np.diff(hist) <= 1e-9), '步长 1/L 损失应单调不增'
acc = np.mean((Xc@w > 0).astype(float) == yc)
print(f'真实数据训练准确率 = {acc:.3f}')
assert acc > 0.9, '真实数据上 logistic 应可分得不错'
print('✅ 真实数据 logistic 上 GD 步长 1/L 单调收敛、准确率高')

**🧪 胶囊练习**：实现 `final_grad_norm(steps)` 返回跑 `steps` 步 GD 后的梯度范数 $\|\nabla(\text{loss})\|$（应随步数减小，趋近驻点）。

In [ ]:
def final_grad_norm(steps):
    # TODO: 从 w=0 跑 steps 步 GD(步长 1/L_C), 返回 np.linalg.norm(ll_grad(w))
    raise NotImplementedError

In [ ]:
# 胶囊自测
def final_grad_norm(steps):
    w = np.zeros(dC)
    for _ in range(steps): w = w - (1/L_C)*ll_grad(w)
    return np.linalg.norm(ll_grad(w))
g500, g4000 = final_grad_norm(500), final_grad_norm(4000)
print(f'梯度范数: 500步={g500:.5f}, 4000步={g4000:.5f}')
assert g4000 < g500, 'GD 应使梯度范数减小(趋近驻点)'
print('✅ 胶囊练习通过：GD 使梯度范数单调趋近 0（趋近驻点）')

In [ ]:
# 📖 胶囊参考答案
def final_grad_norm(steps):
    w = np.zeros(dC)
    for _ in range(steps): w = w - (1/L_C)*ll_grad(w)
    return np.linalg.norm(ll_grad(w))

---
### 小结
- 凸光滑 GD：$O(1/t)$（实测逐点压在 $L\|x_0\|^2/(2t)$ 下）。
- 强凸 GD：线性收敛 $(1-1/\kappa)^t$（指数快），迭代数 $O(\kappa\log\frac1\varepsilon)$。
- SGD：恒定步长停在 ∝ η 的噪声球；$1/(\mu t)$ 步长 = 在线均值，误差 $\sigma^2/T$。
- 动量/Nesterov 把 $\kappa$ 改善成 $\sqrt\kappa$（实测 $\kappa\approx400$ 时 ~20× 加速）；这是一阶下界。
- 非凸：找驻点 $O(1/\sqrt T)$；鞍点是主障碍；PL 条件给非凸也带来线性收敛。

下一站：**模块 03 · NTK 与无限宽**——把非凸的宽网络训练*精确*变成一个凸（核）问题。